# Task 2: read prediction (accuracy ↑)

**Shipped:** pairwise logistic regression on φ(u, item1) − φ(u, item2), with
φ = [BPR, log popularity, cosine sum/max/mean to history], then test-design rules
(unseen item: guard, 0% of rows; seen by user: 1.92% of rows).
**Val 0.9371 (held-out half) | public 0.9407 | tier 4.0** (cutoff 0.94; margin +0.0007).

**Goal now:** margin on the 4.0. Try more similarity measures as extra feature columns (E5).

**Data facts:** decoys popularity-matched; every test candidate appears in training;
the test's decoy sampler does not exclude the user's history (rules.ipynb).

Related: `task2_diagnostics.ipynb` (error analysis), `rules.ipynb` (rule evidence).

In [23]:
%load_ext autoreload
%autoreload 2


from assignment_1.modules.util import calc_pair_accuracy, load_csv, make_pair_validation
from assignment_1.modules.tasks.task2 import TASK2_BPR_PARAMS, task2_pair_features
from assignment_1.modules.models import BPRModel, binary_cosine, jaccard
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
import pandas as pd
import numpy as np
import itertools


DATA_DIR = "../data"  # notebook runs from notebooks/

train = load_csv("tasks_1_2_train.csv", DATA_DIR)
test = load_csv("task_2_test.csv", DATA_DIR)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data checks
Test decoys are popularity-matched if item1/item2 medians match the "∝ interactions" reference.

In [13]:
pop = train["item"].value_counts()
p1, p2 = test["item1"].map(pop).fillna(0), test["item2"].map(pop).fillna(0)
print(f"unseen users {(~test.user.isin(train.user)).mean():.1%} | "
      f"unseen item1 {(p1 == 0).mean():.1%}, item2 {(p2 == 0).mean():.1%}")
print(f"median popularity: item1 {p1.median():.0f}, item2 {p2.median():.0f} | "
      f"refs: uniform {pop.median():.0f}, ∝ interactions {train['item'].map(pop).median():.0f}")
print(
    f"P(item1 more popular) {(p1 > p2).mean():.3f} | ties {(p1 == p2).mean():.3f}")

unseen users 0.0% | unseen item1 0.0%, item2 0.0%
median popularity: item1 58, item2 57 | refs: uniform 18, ∝ interactions 59
P(item1 more popular) 0.497 | ties 0.007


## Validation pairs
Same call as `run_task2`: held-out interactions + popularity-sampled decoys (history excluded).
The combiner is fit on `half` and scored on `~half`.

In [14]:
tr, pairs = make_pair_validation(train, seed=0)
y = (pairs["which"] == pairs["item1"]).astype(int).to_numpy()
half = np.random.default_rng(0).random(len(pairs)) < 0.5
print(f"{len(pairs):,} validation pairs")

186,164 validation pairs


## E1. Reference scorers (full validation)
**Result:** random ≈ 0.50, constant 0.50 (ties → item1), popularity ≈ 0.50 (decoys matched),
baseline BPR (5 factors) ≈ 0.877.

In [15]:
rng = np.random.default_rng(0)
pd.Series({
    "random": calc_pair_accuracy(pairs, lambda u, i: rng.random(len(i))),
    "constant": calc_pair_accuracy(pairs, lambda u, i: np.zeros(len(i))),
    "popularity": calc_pair_accuracy(pairs, lambda u, i: pd.Series(i).map(pop).fillna(0).to_numpy()),
    "BPR 5 factors": calc_pair_accuracy(pairs, BPRModel(
        factors=5, regularization=0.01, learning_rate=0.01, iterations=100).fit(tr)),
}).round(4)

random           0.4987
constant         0.5018
popularity       0.5056
BPR 5 factors    0.8771
dtype: float64

## E2. BPR grid
**Result:** best factors 100, regularization 0.003 → `TASK2_BPR_PARAMS`.
learning_rate / iterations never swept (always 0.01 / 100).

In [16]:
def bpr_grid(grid: dict) -> pd.DataFrame:
    rows = []
    for values in tqdm(list(itertools.product(*grid.values()))):
        params = dict(zip(grid, values))
        rows.append(
            {**params, "val_acc": calc_pair_accuracy(pairs, BPRModel(**params).fit(tr))})
    return pd.DataFrame(rows).sort_values("val_acc", ascending=False).reset_index(drop=True)

# bpr_grid({"factors": [50, 100, 200], "regularization": [0.003, 0.01, 0.03],
#           "learning_rate": [0.01], "iterations": [100]})

## Helper: score a feature matrix with LR and GB on the held-out half

In [17]:
def evaluate(X: pd.DataFrame, cols: list[str] | None = None) -> dict:
    Xc = (X if cols is None else X[cols]).to_numpy()
    out = {}
    for name, m in [("lr", LogisticRegression(max_iter=1000)),
                    ("gb", HistGradientBoostingClassifier(random_state=0))]:
        m.fit(Xc[half], y[half])
        out[name] = round((m.predict(Xc[~half]) == y[~half]).mean(), 4)
    return out

## E3. Pairwise combiner: which features help? (held-out half)
**Result:** BPR alone 0.9146 | BPR + pop 0.9146 (no gain: decoys popularity-matched)
| + cosine LR 0.9371, GB 0.9373 (trees add nothing → ship LR, the course model).

In [18]:
# shipped features (binary cosine), named columns
X = task2_pair_features(tr, pairs)
print(
    f"BPR alone    {((X['bpr'].to_numpy()[~half] >= 0).astype(int) == y[~half]).mean():.4f}")
print("bpr + pop   ", evaluate(X, ["bpr", "log_pop"]))
print("+ cosine    ", evaluate(X))

BPR alone    0.9146
bpr + pop    {'lr': np.float64(0.9146), 'gb': np.float64(0.9138)}
+ cosine     {'lr': np.float64(0.9371), 'gb': np.float64(0.9373)}


## E4. LR coefficients (shipped model)
Sign and size of each feature's weight; features are unscaled, so compare signs, not sizes.

In [19]:
lr = LogisticRegression(max_iter=1000).fit(X.to_numpy()[half], y[half])
pd.Series(lr.coef_[0], index=X.columns).round(3)

bpr          0.909
log_pop     -0.164
cos_sum     -0.013
cos_max     10.097
cos_mean     0.578
dtype: float64

## E5. Similarity measures (held-out half)
Binary cosine (shipped) vs Jaccard vs both as separate column sets. Target: a clear gain over
0.9371 to widen the 4.0 margin (public +0.0007 above 0.94).

In [20]:
variants = {
    "cosine (shipped)": [(binary_cosine, "cos")],
    "jaccard": [(jaccard, "jac")],
    "cosine + jaccard": [(binary_cosine, "cos"), (jaccard, "jac")],
}
for name, sims in variants.items():
    print(f"{name:<18}", evaluate(
        task2_pair_features(tr, pairs, similarities=sims)))

cosine (shipped)   {'lr': np.float64(0.9371), 'gb': np.float64(0.9373)}
jaccard            {'lr': np.float64(0.9362), 'gb': np.float64(0.9369)}
cosine + jaccard   {'lr': np.float64(0.9371), 'gb': np.float64(0.9374)}


In [25]:
Xk = task2_pair_features(tr, pairs, similarities=[(binary_cosine, "cos", (3, 5, 10))])
print(Xk.columns.tolist())
Xk.describe().T[["mean", "std"]]
# print(evaluate(task2_pair_features(tr, pairs, similarities=[(binary_cosine, "cos", (3, 5, 10))])))

['bpr', 'log_pop', 'cos_sum', 'cos_max', 'cos_mean', 'cos_top3', 'cos_top5', 'cos_top10']


,mean,std
bpr,-0.001379,4.218507
log_pop,0.001491,1.921379
cos_sum,-0.016340,10.046414
cos_max,0.000387,0.253659
cos_mean,0.000111,0.042749
cos_top3,0.000233,0.193268
cos_top5,0.000226,0.163256
cos_top10,0.000166,0.126544


## Next
- If a variant clearly beats 0.9371: set it in `TASK2_SIMILARITIES`, `make check`, upload.
- Top-k similarity (mean of the k largest) and recency-weighted history as further columns.
- Validation built like the test (decoys may come from the user's history), so the combiner
  can learn the seen-by-user case instead of relying on the rule.